# 🧠 Tutorial 01: Introducción al Meta-Learning

## "Aprender a Aprender"

Bienvenido al primer tutorial de Meta-Learning! En este notebook aprenderás:

- 📚 Qué es Meta-Learning y por qué es importante
- 🔄 Diferencia entre ML tradicional, Transfer Learning y Meta-Learning
- 🎯 El concepto fundamental de "conjunto de tareas" vs "conjunto de datos"
- 💻 Tu primera implementación práctica de Few-Shot Learning

---

## 📖 Parte 1: Teoría

### ¿Qué es Meta-Learning?

**Meta-Learning** (o "aprender a aprender") es un paradigma de Machine Learning donde el objetivo no es simplemente optimizar parámetros para una tarea específica, sino **optimizar el proceso de aprendizaje mismo**.

#### Analogía del Estudiante:

- **ML Tradicional**: Es como un estudiante que memoriza las respuestas de un examen específico.
- **Transfer Learning**: Es como un estudiante que usa conocimiento de una materia similar.
- **Meta-Learning**: Es como un estudiante que aprende **técnicas de estudio** que le permiten dominar cualquier nueva materia rápidamente.

### Conceptos Clave:

1. **Conjunto de Tareas (Task Set)**: En lugar de entrenar con un dataset, entrenamos con múltiples tareas relacionadas.

2. **Support Set (Soporte)**: Pequeño conjunto de ejemplos para adaptar el modelo a una nueva tarea.

3. **Query Set (Consulta)**: Conjunto de test para evaluar la adaptación.

4. **Few-Shot Learning**: Aprender de pocos ejemplos (1-shot, 5-shot, etc.)

### La Gran Diferencia:

```
ML Tradicional:
  Datos (muchos) → Modelo → Predicciones en la misma distribución

Meta-Learning:
  Múltiples Tareas → Meta-Modelo → Adaptación rápida a NUEVAS tareas
```


---

## 🛠️ Parte 2: Setup y Librerías

In [ ]:
# Importar librerías necesarias
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from utils.test_utils import (
    print_success, print_hint, HintSystem, 
    check_implementation, run_test
)
from utils.data_utils import create_sine_task, set_seed
from utils.visualization import plot_few_shot_results

# Configurar semilla para reproducibilidad
set_seed(42)

# Verificar si GPU está disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Usando dispositivo: {device}")
print("✅ Librerías importadas correctamente!")

---

## 📊 Parte 3: Visualizando el Problema

Vamos a crear nuestro primer ejemplo de Meta-Learning: **regresión de funciones sinusoidales**.

Cada tarea será aprender una función seno con amplitud y fase diferentes:
$$y = A \sin(x + \phi)$$

In [ ]:
# Generar una tarea de ejemplo
task = create_sine_task(k_shot=10, q_query=50)

print(f"📊 Tarea generada:")
print(f"  Amplitud: {task['amplitude']:.2f}")
print(f"  Fase: {task['phase']:.2f}")
print(f"  Support set: {task['x_support'].shape}")
print(f"  Query set: {task['x_query'].shape}")

# Visualizar
plot_few_shot_results(
    task['x_support'], task['y_support'],
    task['x_query'], task['y_query'],
    title="Ejemplo de Tarea de Meta-Learning: Regresión Sinusoidal"
)

### 🤔 Observación Importante:

Nota que tenemos **solo 10 ejemplos de soporte** (puntos azules). 

- ❌ **ML Tradicional**: Necesitaría cientos o miles de puntos para aprender bien
- ✅ **Meta-Learning**: Puede adaptarse con muy pocos ejemplos si ha visto tareas similares antes

---

## 💻 Parte 4: Ejercicio 1 - Modelo Simple de Regresión

Vamos a crear un modelo de red neuronal simple para regresión.

**Tu tarea**: Completa la implementación del modelo.

In [ ]:
class SimpleRegressionModel(nn.Module):
    """
    Red neuronal simple para regresión.
    
    Arquitectura:
        Input (1D) → Hidden (40) → Hidden (40) → Output (1D)
    """
    
    def __init__(self, input_dim=1, hidden_dim=40, output_dim=1):
        super(SimpleRegressionModel, self).__init__()
        
        # TODO: Define las capas de la red
        # Necesitas:
        # 1. Una capa lineal de input_dim a hidden_dim
        # 2. Una capa lineal de hidden_dim a hidden_dim
        # 3. Una capa lineal de hidden_dim a output_dim
        
        self.fc1 = None  # TODO: Reemplaza None con nn.Linear(...)
        self.fc2 = None  # TODO: Reemplaza None con nn.Linear(...)
        self.fc3 = None  # TODO: Reemplaza None con nn.Linear(...)
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input tensor [batch_size, input_dim]
        
        Returns:
            output: Predicciones [batch_size, output_dim]
        """
        # TODO: Implementa el forward pass
        # Usa activación ReLU entre capas
        # Recuerda: x -> fc1 -> ReLU -> fc2 -> ReLU -> fc3 -> output
        
        pass  # TODO: Elimina esta línea y escribe tu código


# Sistema de pistas
hints_model = HintSystem([
    "Usa nn.Linear(in_features, out_features) para crear capas lineales.",
    "En el forward pass, usa torch.relu() o F.relu() para activaciones.",
    "La estructura es: x = relu(fc1(x)), x = relu(fc2(x)), x = fc3(x)",
    "Solución completa: self.fc1 = nn.Linear(input_dim, hidden_dim), similar para fc2 y fc3"
])

In [ ]:
# Para ver pistas, ejecuta esta celda múltiples veces
hints_model.show_hint()

In [ ]:
# ✅ TEST 1: Verificar que el modelo se construye correctamente

def test_model_construction():
    model = SimpleRegressionModel()
    
    # Verificar que las capas existen
    assert hasattr(model, 'fc1'), "El modelo debe tener una capa fc1"
    assert hasattr(model, 'fc2'), "El modelo debe tener una capa fc2"
    assert hasattr(model, 'fc3'), "El modelo debe tener una capa fc3"
    
    # Verificar que no son None
    assert model.fc1 is not None, "fc1 no debe ser None"
    assert model.fc2 is not None, "fc2 no debe ser None"
    assert model.fc3 is not None, "fc3 no debe ser None"
    
    # Verificar dimensiones
    assert model.fc1.in_features == 1, "fc1 debe tener input_dim=1"
    assert model.fc1.out_features == 40, "fc1 debe tener output=40"
    assert model.fc3.out_features == 1, "fc3 debe tener output_dim=1"
    
    print_success("✅ Modelo construido correctamente!")

run_test(test_model_construction, "Test de Construcción del Modelo")

In [ ]:
# ✅ TEST 2: Verificar que el forward pass funciona

def test_forward_pass():
    model = SimpleRegressionModel()
    
    # Crear input de prueba
    x = torch.randn(5, 1)  # Batch de 5 ejemplos
    
    # Forward pass
    output = model(x)
    
    # Verificar shape
    assert output.shape == (5, 1), f"Output debe tener shape (5, 1), pero tiene {output.shape}"
    
    # Verificar que no hay NaN o Inf
    assert not torch.isnan(output).any(), "Output contiene NaN"
    assert not torch.isinf(output).any(), "Output contiene Inf"
    
    print_success("✅ Forward pass funciona correctamente!")

run_test(test_forward_pass, "Test de Forward Pass")

---

## 💻 Parte 5: Ejercicio 2 - Entrenamiento Simple en Una Tarea

Ahora vamos a entrenar el modelo en una sola tarea usando gradient descent tradicional.

**Tu tarea**: Completa la función de entrenamiento.

In [ ]:
def train_on_task(model, task, n_steps=100, lr=0.01):
    """
    Entrena el modelo en una tarea específica.
    
    Args:
        model: Modelo a entrenar
        task: Diccionario con x_support, y_support
        n_steps: Número de pasos de gradient descent
        lr: Learning rate
    
    Returns:
        losses: Lista de pérdidas durante el entrenamiento
    """
    # TODO: Crea un optimizador SGD con learning rate lr
    optimizer = None  # TODO: Usa optim.SGD(model.parameters(), lr=lr)
    
    # Criterio de pérdida (MSE para regresión)
    criterion = nn.MSELoss()
    
    # Extraer datos de soporte
    x_support = task['x_support']
    y_support = task['y_support']
    
    losses = []
    
    for step in range(n_steps):
        # TODO: Implementa el loop de entrenamiento
        # 1. Zero gradients: optimizer.zero_grad()
        # 2. Forward pass: predictions = model(x_support)
        # 3. Calcular loss: loss = criterion(predictions, y_support)
        # 4. Backward pass: loss.backward()
        # 5. Update parameters: optimizer.step()
        # 6. Guardar loss: losses.append(loss.item())
        
        pass  # TODO: Elimina esta línea y escribe tu código
    
    return losses


# Sistema de pistas
hints_training = HintSystem([
    "El loop de entrenamiento sigue el patrón estándar: zero_grad → forward → loss → backward → step.",
    "optimizer.zero_grad() limpia los gradientes, loss.backward() calcula gradientes, optimizer.step() actualiza parámetros.",
    "No olvides llamar loss.item() para obtener el valor numérico del loss (en lugar del tensor).",
    "Estructura: optimizer.zero_grad(); pred = model(x); loss = criterion(pred, y); loss.backward(); optimizer.step(); losses.append(loss.item())"
])

In [ ]:
# Para ver pistas
hints_training.show_hint()

In [ ]:
# ✅ TEST 3: Verificar que el entrenamiento funciona

def test_training():
    model = SimpleRegressionModel()
    task = create_sine_task(k_shot=20, q_query=10)
    
    losses = train_on_task(model, task, n_steps=50, lr=0.01)
    
    # Verificar que devuelve una lista
    assert isinstance(losses, list), "train_on_task debe devolver una lista"
    assert len(losses) == 50, "Debe haber 50 valores de loss"
    
    # Verificar que el loss disminuye
    assert losses[-1] < losses[0], "El loss debe disminuir durante el entrenamiento"
    
    print_success(f"✅ Entrenamiento funciona! Loss inicial: {losses[0]:.4f}, Loss final: {losses[-1]:.4f}")

run_test(test_training, "Test de Entrenamiento")

---

## 📊 Parte 6: Visualizar Resultados

Ahora vamos a entrenar el modelo y visualizar sus predicciones.

In [ ]:
# Crear una nueva tarea
task = create_sine_task(k_shot=10, q_query=50)

# Crear y entrenar modelo
model = SimpleRegressionModel()
losses = train_on_task(model, task, n_steps=200, lr=0.01)

# Hacer predicciones
model.eval()
with torch.no_grad():
    y_pred = model(task['x_query'])

# Visualizar
plot_few_shot_results(
    task['x_support'], task['y_support'],
    task['x_query'], task['y_query'],
    y_pred=y_pred,
    title="Resultados después de Entrenamiento (200 pasos)"
)

# Plot de la curva de aprendizaje
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Paso de Entrenamiento')
plt.ylabel('Loss (MSE)')
plt.title('Curva de Aprendizaje')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n📊 Métricas Finales:")
print(f"  Loss inicial: {losses[0]:.4f}")
print(f"  Loss final: {losses[-1]:.4f}")
print(f"  Reducción: {(1 - losses[-1]/losses[0])*100:.1f}%")

---

## 🎯 Parte 7: El Problema del ML Tradicional

El enfoque que acabas de implementar tiene una **limitación crítica**:

### ❌ Problema:
- Necesitas entrenar **desde cero** para cada nueva tarea
- Requieres **muchos pasos de gradient descent**
- No aprovechas el conocimiento de tareas anteriores

### ✅ Solución: Meta-Learning
En los próximos tutoriales aprenderás cómo:
1. Entrenar en múltiples tareas simultáneamente
2. Aprender una **inicialización óptima** que permite adaptación rápida
3. Usar algoritmos como **MAML** que logran adaptación en 1-5 pasos

### Experimento:
Ejecuta la siguiente celda para ver cómo un modelo sin entrenar falla completamente:

In [ ]:
# Modelo sin entrenar (inicialización aleatoria)
untrained_model = SimpleRegressionModel()

# Nueva tarea
new_task = create_sine_task(k_shot=5, q_query=50)

# Predicción sin entrenamiento
untrained_model.eval()
with torch.no_grad():
    untrained_pred = untrained_model(new_task['x_query'])

# Modelo entrenado en la tarea
trained_model = SimpleRegressionModel()
_ = train_on_task(trained_model, new_task, n_steps=100, lr=0.01)
trained_model.eval()
with torch.no_grad():
    trained_pred = trained_model(new_task['x_query'])

# Visualizar comparación
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Sin entrenar
ax1.scatter(new_task['x_support'], new_task['y_support'], c='blue', s=100, label='Support', zorder=3)
ax1.scatter(new_task['x_query'], new_task['y_query'], c='green', s=50, alpha=0.6, label='True Query')
ax1.scatter(new_task['x_query'], untrained_pred, c='red', s=50, alpha=0.6, label='Predicted', marker='^')
ax1.set_title('❌ Modelo Sin Entrenar (Inicialización Aleatoria)', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Entrenado
ax2.scatter(new_task['x_support'], new_task['y_support'], c='blue', s=100, label='Support', zorder=3)
ax2.scatter(new_task['x_query'], new_task['y_query'], c='green', s=50, alpha=0.6, label='True Query')
ax2.scatter(new_task['x_query'], trained_pred, c='red', s=50, alpha=0.6, label='Predicted', marker='^')
ax2.set_title('✅ Modelo Entrenado (100 pasos de GD)', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎯 Observación clave:")
print("El modelo sin entrenar hace predicciones completamente aleatorias.")
print("Necesitamos 100+ pasos de entrenamiento para cada nueva tarea.")
print("\n💡 En el siguiente tutorial veremos cómo Meta-Learning soluciona esto!")

---

## 🎓 Resumen y Conclusiones

### ✅ Lo que aprendiste:

1. **Meta-Learning** es sobre aprender el proceso de aprendizaje, no solo parámetros
2. Trabajamos con **conjuntos de tareas**, no conjuntos de datos
3. **Support set** = ejemplos para adaptación, **Query set** = evaluación
4. ML tradicional necesita entrenar desde cero para cada tarea

### 🚀 Próximos Pasos:

En el siguiente tutorial (**02_curva_aprendizaje.ipynb**) compararemos:
- ML Tradicional vs Transfer Learning vs Meta-Learning
- Velocidades de adaptación
- Eficiencia en pocos ejemplos

### 📚 Recursos Adicionales:

- Paper: [Learning to Learn](https://link.springer.com/chapter/10.1007/978-1-4615-5529-2_5) - Thrun & Pratt, 1998
- Blog: [Meta-Learning Explained](https://lilianweng.github.io/posts/2018-11-30-meta-learning/)

---

## 🎉 ¡Felicidades!

Has completado el primer tutorial de Meta-Learning. Ahora entiendes los conceptos fundamentales y has implementado tu primer sistema de aprendizaje adaptativo básico.

**🔥 Desafío Opcional:** Intenta modificar la arquitectura del modelo (más capas, más neuronas) y observa cómo afecta el entrenamiento.
